# MIB-Hub: Cloud GPU Asset Pipeline (Blender + MCP)

**What this notebook does:**
1. Installs Blender 4.0 (headless, optimized for Colab GPU)
2. Installs MPFB2 (MakeHuman parametric character generator)
3. Mounts your Google Drive for persistent storage
4. Starts an MCP server so Claude can control Blender remotely
5. Exposes the server via ngrok (public HTTPS tunnel)

**Runtime:** GPU (T4 recommended, free tier)

**How to use:**
1. Click **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order
3. Copy the ngrok URL from the output
4. Give that URL to Claude

---

## 1️⃣ Mount Google Drive (Persistent Storage)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/MIB-Hub-Assets'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/characters', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/buildings', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/exports', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/screenshots', exist_ok=True)

print(f"✅ Google Drive mounted at: {PROJECT_DIR}")
print(f"📂 All .blend files and exports will be saved here (persistent across sessions)")

## 2️⃣ Install Blender 4.0 LTS (Headless)

In [ ]:
%%bash
# Install system dependencies
apt-get update -qq
apt-get install -y -qq wget xz-utils libxrender1 libxi6 libxkbcommon0 libsm6 > /dev/null 2>&1

# Download and extract Blender 4.0.2 LTS
cd /content
wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
tar -xf blender-4.0.2-linux-x64.tar.xz
rm blender-4.0.2-linux-x64.tar.xz

# Create symlink
ln -sf /content/blender-4.0.2-linux-x64/blender /usr/local/bin/blender

# Verify installation
blender --version

## 3️⃣ Install MPFB2 (MakeHuman Character Generator for Blender)

In [ ]:
%%bash
# Download MPFB2 add-on
cd /content
wget -q https://github.com/makehumancommunity/mpfb2/archive/refs/heads/main.zip -O mpfb2.zip
unzip -q mpfb2.zip
rm mpfb2.zip

# Install add-on into Blender's scripts folder
BLENDER_ADDONS=/content/blender-4.0.2-linux-x64/4.0/scripts/addons
mkdir -p $BLENDER_ADDONS
cp -r mpfb2-main/mpfb $BLENDER_ADDONS/

# Enable the add-on programmatically
blender --background --python-expr "
import bpy
bpy.ops.preferences.addon_enable(module='mpfb')
bpy.ops.wm.save_userpref()
print('✅ MPFB2 add-on enabled')
"

## 4️⃣ Install Python Dependencies for MCP Server

In [ ]:
!pip install -q fastapi uvicorn python-multipart pydantic nest-asyncio pyngrok
print("✅ MCP server dependencies installed")

## 5️⃣ Create MCP Server Script

In [ ]:
%%writefile /content/mcp_blender_server.py
#!/usr/bin/env python3
"""
MCP Server for Headless Blender on Google Colab
Allows Claude to send Blender Python scripts and retrieve rendered outputs.
"""
import asyncio
import json
import os
import subprocess
import tempfile
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel
import nest_asyncio

# Allow nested event loops in Colab
nest_asyncio.apply()

# ============================================================================
# Config
# ============================================================================

BLENDER_BIN = "/usr/local/bin/blender"
WORKSPACE_DIR = Path("/content/drive/MyDrive/MIB-Hub-Assets")
JOBS_DIR = WORKSPACE_DIR / "blender_jobs"
JOBS_DIR.mkdir(exist_ok=True)

jobs: Dict[str, Dict[str, Any]] = {}

# ============================================================================
# Models
# ============================================================================

class BlenderJobRequest(BaseModel):
    script: str
    description: str
    output_format: str = "png"
    render: bool = False
    camera_position: Optional[List[float]] = None
    camera_target: Optional[List[float]] = None

class BlenderJobResponse(BaseModel):
    job_id: str
    status: str
    output_files: List[str]
    stdout: str
    stderr: str
    duration_seconds: float

# ============================================================================
# FastAPI App
# ============================================================================

app = FastAPI(title="Blender MCP Server (Colab)", version="1.0.0")

async def run_blender_script(
    job_id: str,
    script: str,
    output_format: str,
    render: bool,
    camera_position: Optional[List[float]],
    camera_target: Optional[List[float]],
) -> BlenderJobResponse:
    """Execute a Blender Python script."""
    job_dir = JOBS_DIR / job_id
    job_dir.mkdir(exist_ok=True)

    script_path = job_dir / "script.py"
    output_path = job_dir / f"output.{output_format}"

    # Wrap script with output setup
    wrapped_script = f"""
import bpy
import time

start_time = time.time()

# User script
{script}
"""

    if camera_position and camera_target:
        wrapped_script += f"""
# Setup camera
import mathutils
if 'Camera' not in bpy.data.objects:
    bpy.ops.object.camera_add(location={camera_position})
    cam = bpy.context.object
    cam.name = 'Camera'
else:
    cam = bpy.data.objects['Camera']
    cam.location = {camera_position}

direction = mathutils.Vector({camera_target}) - mathutils.Vector({camera_position})
rot_quat = direction.to_track_quat('-Z', 'Y')
cam.rotation_euler = rot_quat.to_euler()
bpy.context.scene.camera = cam
"""

    if render:
        wrapped_script += f"""
# Render
bpy.context.scene.render.filepath = '{output_path.as_posix()}'
bpy.context.scene.render.image_settings.file_format = '{output_format.upper()}'
bpy.context.scene.render.resolution_x = 1920
bpy.context.scene.render.resolution_y = 1080
bpy.context.scene.render.engine = 'CYCLES'
bpy.context.scene.cycles.samples = 64
bpy.context.scene.cycles.device = 'GPU'
bpy.ops.render.render(write_still=True)
print(f"RENDERED: {output_path.as_posix()}")
"""
    elif output_format == "glb":
        wrapped_script += f"""
# Export GLB
bpy.ops.export_scene.gltf(
    filepath='{output_path.as_posix()}',
    export_format='GLB',
    export_draco_mesh_compression_enable=True,
    export_draco_mesh_compression_level=6
)
print(f"EXPORTED: {output_path.as_posix()}")
"""
    elif output_format == "blend":
        wrapped_script += f"""
bpy.ops.wm.save_as_mainfile(filepath='{output_path.as_posix()}')
print(f"SAVED: {output_path.as_posix()}")
"""

    wrapped_script += """
duration = time.time() - start_time
print(f"DURATION: {duration:.2f}s")
"""

    script_path.write_text(wrapped_script)

    # Run Blender
    cmd = [BLENDER_BIN, "--background", "--python", str(script_path)]

    proc = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        cwd=str(job_dir),
    )

    stdout, stderr = await proc.communicate()
    stdout_text = stdout.decode("utf-8")
    stderr_text = stderr.decode("utf-8")

    # Parse duration
    duration = 0.0
    for line in stdout_text.splitlines():
        if line.startswith("DURATION:"):
            duration = float(line.split(":")[1].strip().rstrip("s"))

    # Find output files
    output_files = [str(f.relative_to(WORKSPACE_DIR)) for f in job_dir.glob("output.*")]

    status = "completed" if proc.returncode == 0 else "failed"

    return BlenderJobResponse(
        job_id=job_id,
        status=status,
        output_files=output_files,
        stdout=stdout_text,
        stderr=stderr_text,
        duration_seconds=duration,
    )

# ============================================================================
# API Endpoints
# ============================================================================

@app.post("/mcp/jobs", response_model=BlenderJobResponse)
async def create_blender_job(req: BlenderJobRequest):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "running", "request": req.dict()}

    result = await run_blender_script(
        job_id=job_id,
        script=req.script,
        output_format=req.output_format,
        render=req.render,
        camera_position=req.camera_position,
        camera_target=req.camera_target,
    )

    jobs[job_id] = result.dict()
    return result

@app.get("/mcp/jobs/{job_id}", response_model=BlenderJobResponse)
async def get_job_status(job_id: str):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")
    return JSONResponse(content=jobs[job_id])

@app.get("/mcp/jobs/{job_id}/download/{filename}")
async def download_output(job_id: str, filename: str):
    file_path = JOBS_DIR / job_id / filename
    if not file_path.exists():
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(file_path)

@app.get("/mcp/health")
async def health_check():
    blender_version = subprocess.run(
        [BLENDER_BIN, "--version"],
        capture_output=True,
        text=True,
    ).stdout.split("\n")[0]

    return {
        "status": "ok",
        "blender": blender_version,
        "workspace": str(WORKSPACE_DIR),
        "jobs_count": len(jobs),
    }

print("✅ MCP Server ready")


## 6️⃣ Start MCP Server + Ngrok Tunnel

In [ ]:
import threading
import time
from pyngrok import ngrok
import uvicorn

# Start ngrok tunnel
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")  # Get free token from https://ngrok.com
public_url = ngrok.connect(8080, bind_tls=True)

print("="*80)
print("🚀 MCP SERVER IS LIVE!")
print("="*80)
print(f"\n📡 Public URL: {public_url}\n")
print("📋 COPY THIS URL AND GIVE IT TO CLAUDE:\n")
print(f"   {public_url}")
print("\n💻 On your LOCAL machine, run:")
print(f"   claude mcp add --transport http blender-colab {public_url}")
print("\n⏳ Server will keep running until you stop this cell...\n")
print("="*80)

# Start server in background thread
def start_server():
    uvicorn.run(
        "mcp_blender_server:app",
        host="0.0.0.0",
        port=8080,
        log_level="info"
    )

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Keep cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Server stopped")